# Feature Engineering

Tier A/B feature builders (Decision #1), target definition/transforms (Decisions #2, #3), failed-job exclusion (Decision #4), embedding dimensionality reduction (Decision #7).

See the conceptualization plan and EXPERIMENT_TRACKER.md for full context.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd

from src import config, features, metrics, plotting, models, roofline


## Load data

Same `SAMPLE_SIZE` dev-sample pattern as notebook 01 — see `EXPERIMENT_TRACKER.md` for when this needs to step up toward the full dataset.

In [2]:
FDATA_DIR = "../data/raw/fdata"
PM100_PATH = "../data/raw/pm100/pm100_job_table.parquet"
SAMPLE_SIZE = 5000

import glob
fdata_files = sorted(glob.glob(f"{FDATA_DIR}/*.parquet"))
fdata = pd.concat([pd.read_parquet(f) for f in fdata_files], ignore_index=True)
pm100 = pd.read_parquet(PM100_PATH)

if SAMPLE_SIZE is not None:
    fdata = fdata.sample(n=min(SAMPLE_SIZE, len(fdata)), random_state=0).reset_index(drop=True)
    pm100 = pm100.sample(n=min(SAMPLE_SIZE, len(pm100)), random_state=0).reset_index(drop=True)

FDATA_DATETIME_COLS = ["adt", "qdt", "schedsdt", "deldt", "sdt", "edt"]
for c in FDATA_DATETIME_COLS:
    fdata[c] = pd.to_datetime(fdata[c], errors="coerce")
pm100["submit_time"] = pd.to_datetime(pm100["submit_time"], unit="s", errors="coerce")
pm100["start_time"] = pd.to_datetime(pm100["start_time"], unit="s", errors="coerce")
pm100["end_time"] = pd.to_datetime(pm100["end_time"], unit="s", errors="coerce")

print("fdata:", fdata.shape, " pm100:", pm100.shape)


fdata: (5000, 45)  pm100: (5000, 35)


## Step 1: Exclude failed/cancelled/killed jobs (Decision #4)

Their duration/power are artifacts of failure, not real workload behavior. F-DATA's `exit state` is a clean completed/failed binary; PM100's `job_state` has 6 values, only `COMPLETED` reflects normal execution.

In [3]:
print("F-DATA exit state breakdown:")
print(fdata["exit state"].value_counts())
print("\nPM100 job_state breakdown:")
print(pm100["job_state"].value_counts())

fdata = features.filter_completed_jobs(fdata, "fdata")
pm100 = features.filter_completed_jobs(pm100, "pm100")


F-DATA exit state breakdown:
exit state
completed    4743
failed        257
Name: count, dtype: int64

PM100 job_state breakdown:
job_state
COMPLETED        3952
FAILED            627
CANCELLED         208
TIMEOUT           196
OUT_OF_MEMORY      16
NODE_FAIL           1
Name: count, dtype: int64
[fdata] excluded 5.1% of jobs as non-completed (257 / 5000)
[pm100] excluded 21.0% of jobs as non-completed (1048 / 5000)


## Step 2: Tier A/B feature matrices (Decision #1)

Re-running the leakage sanity check from notebook 01 on the post-exclusion data — this should still pass, but re-verifying after every transformation step is the point of Decision #19.

In [4]:
fdata_tier_a = features.build_tier_a_features(fdata, "fdata")
fdata_tier_b = features.build_tier_b_features(fdata, "fdata")
pm100_tier_a = features.build_tier_a_features(pm100, "pm100")
pm100_tier_b = features.build_tier_b_features(pm100, "pm100")

for name, cols, tier, dataset in [
    ("F-DATA Tier A", fdata_tier_a.columns, "A", "fdata"),
    ("F-DATA Tier B", fdata_tier_b.columns, "B", "fdata"),
    ("PM100 Tier A", pm100_tier_a.columns, "A", "pm100"),
    ("PM100 Tier B", pm100_tier_b.columns, "B", "pm100"),
]:
    features.assert_no_tier_leakage(list(cols), tier, dataset)
    print(f"{name}: {len(cols)} columns, OK")


F-DATA Tier A: 13 columns, OK
F-DATA Tier B: 30 columns, OK
PM100 Tier A: 18 columns, OK
PM100 Tier B: 15 columns, OK


## Step 3: Embedding dimensionality reduction (Decision #7)

F-DATA only — PCA-reduce the 384-dim SBert `embedding` column to 10 components for the tree-based models. PM100 has no job-name embedding field.

In [5]:
fdata_emb_pca = features.reduce_fdata_embedding(fdata, n_components=10)
print(fdata_emb_pca.shape)
fdata_emb_pca.describe().T


(4743, 10)


,count,mean,std,min,25%,50%,75%,max
emb_pc_0,4743.0,-7.039455e-07,0.378917,-0.484775,-0.353825,-0.145981,0.469826,0.476844
emb_pc_1,4743.0,-4.154103e-07,0.169386,-0.485055,-0.015524,0.024949,0.074911,0.401953
emb_pc_2,4743.0,-6.151732e-07,0.144977,-0.486339,-0.075051,0.012000,0.038803,0.357505
emb_pc_3,4743.0,-4.741227e-07,0.119908,-0.410849,-0.043893,-0.003434,0.003393,0.497290
emb_pc_4,4743.0,-2.528202e-07,0.112464,-0.251344,-0.050743,-0.021371,0.047927,0.364435
emb_pc_5,4743.0,1.494703e-07,0.107177,-0.485642,-0.038458,-0.030617,0.071606,0.309108
emb_pc_6,4743.0,2.640550e-07,0.104905,-0.186583,-0.068091,0.017020,0.056123,0.468203
emb_pc_7,4743.0,-3.844456e-07,0.098583,-0.305955,-0.067658,0.006920,0.024882,0.370750
emb_pc_8,4743.0,4.996335e-07,0.097027,-0.305378,-0.034318,0.000894,0.013995,0.349703
emb_pc_9,4743.0,5.567122e-08,0.085038,-0.338255,-0.045575,0.009274,0.037362,0.298719


## Step 4: Historical rolling-stat features (Tier A)

Each user's mean `duration` over their *previous* jobs only (never the current job's own outcome — computed via `shift(1)` before the rolling window, per Decision #1's leakage discipline). With only 1 of F-DATA's 38 months present and a 5000-row dev sample, expect these to be sparse/noisy — this is expected, not a bug (see EXPERIMENT_TRACKER.md).

In [6]:
fdata_with_rolling = features.add_user_rolling_stat(fdata, "fdata", "duration", window=5)
pm100_with_rolling = features.add_user_rolling_stat(pm100, "pm100", "run_time", window=5)

n_fdata_available = fdata_with_rolling["duration_user_rolling_mean"].notna().sum()
n_pm100_available = pm100_with_rolling["run_time_user_rolling_mean"].notna().sum()
print(f"F-DATA: rolling stat available for {n_fdata_available}/{len(fdata_with_rolling)} jobs")
print(f"PM100: rolling stat available for {n_pm100_available}/{len(pm100_with_rolling)} jobs")

fdata_with_rolling[["usr", "adt", "duration", "duration_user_rolling_mean"]].head(10)


F-DATA: rolling stat available for 4586/4743 jobs
PM100: rolling stat available for 3717/3952 jobs


,usr,adt,duration,duration_user_rolling_mean
1,usr_1226,2022-01-01 23:33:34+09:00,446.0,1116.4
2,usr_1365,2022-01-31 20:57:51+09:00,209.0,128.2
4,usr_623,2022-01-13 14:18:12+09:00,8120.0,904.0
5,usr_896,2022-01-26 13:10:04+09:00,108.0,124.0
6,usr_1012,2022-01-23 16:19:22+09:00,1.0,1.2
7,usr_617,2022-01-06 10:40:53+09:00,189.0,180.0
8,usr_1226,2022-01-01 20:51:02+09:00,133.0,1696.2
9,usr_1122,2022-01-13 14:15:28+09:00,529.0,243.0
10,usr_147,2022-01-26 19:01:16+09:00,25.0,65.2
11,usr_1226,2022-01-01 10:39:25+09:00,1164.0,887.8


## Step 5: Target transforms (Decision #3)

log1p on all three targets per dataset (execution time, memory, power) — training happens in log-space, metrics get reported in both log-space and back-transformed real units. Sanity-checking the round-trip here (Decision #19) rather than assuming it.

In [7]:
for name, targets, df in [("F-DATA", features.FDATA_TARGETS, fdata), ("PM100", features.PM100_TARGETS, pm100)]:
    for target_name, col in targets.items():
        raw = df[col].dropna().to_numpy()
        if len(raw) and hasattr(raw[0], "__len__"):
            # PM100 power: array-valued, reduce to per-job mean first (see notebook 01)
            raw = np.array([np.mean(a) for a in raw])
        raw = raw[raw >= 0].astype(float)
        transformed = features.transform_target(raw)
        recovered = features.inverse_transform_target(transformed)
        metrics.expm1_round_trip_check(raw)
        print(f"{name} {target_name} ({col}): round-trip OK, n={len(raw)}, "
              f"raw range [{raw.min():.2f}, {raw.max():.2f}], "
              f"log1p range [{transformed.min():.2f}, {transformed.max():.2f}]")


F-DATA execution_time (duration): round-trip OK, n=4743, raw range [1.00, 234313.00], log1p range [0.69, 12.36]
F-DATA memory (mmszu): round-trip OK, n=4743, raw range [24444928.00, 29955457024.00], log1p range [17.01, 24.12]
F-DATA power (avgpcon): round-trip OK, n=4743, raw range [39.65, 229976.49], log1p range [3.71, 12.35]
PM100 execution_time (run_time): round-trip OK, n=3952, raw range [0.00, 86328.00], log1p range [0.00, 11.37]
PM100 memory (mem_alloc): round-trip OK, n=3952, raw range [1.00, 60800.00], log1p range [0.69, 11.02]
PM100 power (node_power_consumption): round-trip OK, n=3952, raw range [480.00, 173911.93], log1p range [6.18, 12.07]


## Summary

This notebook validates the feature-engineering pipeline end to end on the dev sample: exclusion, Tier A/B separation, embedding reduction, rolling-stat features, and target transforms all pass their sanity checks. Nothing here is yet the "final" feature matrix used for training — that assembly happens in notebooks 05-07, reusing these same `src/features.py` functions against the full dataset once available.